# Training and fine tuning

In [5]:
import optuna
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error

### baseline model

In [7]:
#load csv
df = pd.read_csv('../data/processed/processed_data.csv')

In [9]:
df_base = df.sort_values(by=['resto_name'])

In [14]:
# Using shift(1) is absolutely necessary to prevent data leakage
# (it ensures we only use the *past* 7 days to predict *today*)
baseline_cols = ['breakfast', 'launch', 'dinner']
baseline_pred_cols = [f"{col}_baseline_7d_avg" for col in baseline_cols]

df_base[baseline_pred_cols] = (
    df_base.groupby('resto_name')[baseline_cols]
    .transform(lambda x: x.rolling(window=7, min_periods=1).mean().shift(1))
)

In [15]:
split_index = int(len(df_base) * 0.8)
test_df = df_base.iloc[split_index:].copy()

In [16]:
#Drop rows where the baseline is NaN 
test_clean = test_df.dropna(subset=baseline_pred_cols + baseline_cols)

In [17]:
mae_baseline = {}
rmse_baseline = {}

for col, pred_col in zip(baseline_cols, baseline_pred_cols):
    mae_baseline[col] = mean_absolute_error(test_clean[col], test_clean[pred_col])
    rmse_baseline[col] = np.sqrt(mean_squared_error(test_clean[col], test_clean[pred_col]))

In [18]:
print("--- 7-Day Moving Average Baseline ---")
for col in baseline_cols:
    print(f"{col.title()} MAE:  {mae_baseline[col]:.2f} meals")
    print(f"{col.title()} RMSE: {rmse_baseline[col]:.2f} meals")

--- 7-Day Moving Average Baseline ---
Breakfast MAE:  0.75 meals
Breakfast RMSE: 1.28 meals
Launch MAE:  0.50 meals
Launch RMSE: 0.90 meals
Dinner MAE:  0.50 meals
Dinner RMSE: 0.89 meals


### Catboost

In [19]:
target_cols = ['breakfast', 'launch', 'dinner']
X = df.drop(columns=target_cols)
y = df[target_cols]

In [20]:
split_index = int(len(df) * 0.8)

X_train, X_test = X.iloc[:split_index], X.iloc[split_index:]
y_train, y_test = y.iloc[:split_index], y.iloc[split_index:]

In [21]:
categorical_features = ['resto_name',]

In [22]:
from sklearn.multioutput import MultiOutputRegressor
base_model = CatBoostRegressor(
    iterations=1000,
    learning_rate=0.05,
    depth=6,
    loss_function='MAE', 
    cat_features=categorical_features,
    random_seed=42,
    verbose=False,
    task_type='GPU')

In [23]:
models = {}

print("Training models via MultiOutputRegressor...")
# Under the hood, this trains 3 completely separate CatBoost models!
multi_target_model.fit(X_train, y_train,
    early_stopping_rounds=50,  
    use_best_model=True)

Training models with tqdm...


Training targets:   0%|          | 0/3 [00:00<?, ?it/s]You should provide test set for use best model. use_best_model parameter has been switched to false value.
Default metric period is 5 because MAE is/are not implemented for GPU
Training targets:  33%|███▎      | 1/3 [00:17<00:34, 17.45s/it]You should provide test set for use best model. use_best_model parameter has been switched to false value.
Default metric period is 5 because MAE is/are not implemented for GPU
Training targets:  67%|██████▋   | 2/3 [00:36<00:18, 18.46s/it]You should provide test set for use best model. use_best_model parameter has been switched to false value.
Default metric period is 5 because MAE is/are not implemented for GPU
Training targets: 100%|██████████| 3/3 [01:03<00:00, 21.13s/it]


In [25]:
preds = np.column_stack([models[target].predict(X_test) for target in target_cols])

mae_b = mean_absolute_error(y_test['breakfast'], preds[:, 0])
mae_l = mean_absolute_error(y_test['launch'], preds[:, 1])
mae_d = mean_absolute_error(y_test['dinner'], preds[:, 2])

print(f"Breakfast MAE: {mae_b:.2f}")
print(f"Lunch MAE:     {mae_l:.2f}")
print(f"Dinner MAE:    {mae_d:.2f}")

Breakfast MAE: 0.25
Lunch MAE:     0.17
Dinner MAE:    0.17


In [ ]:
import matplotlib.pyplot as plt

pred_df = pd.DataFrame(preds, columns=target_cols, index=y_test.index)

fig, axes = plt.subplots(1, len(target_cols), figsize=(5 * len(target_cols), 4))
if len(target_cols) == 1:
    axes = [axes]

for ax, col in zip(axes, target_cols):
    ax.scatter(y_test[col], pred_df[col], s=12, alpha=0.6)
    min_val = min(y_test[col].min(), pred_df[col].min())
    max_val = max(y_test[col].max(), pred_df[col].max())
    ax.plot([min_val, max_val], [min_val, max_val], color="red", linewidth=1)
    ax.set_title(f"{col.title()}: Pred vs Actual")
    ax.set_xlabel("Actual")
    ax.set_ylabel("Predicted")

plt.tight_layout()
plt.show()